# Deep Learning-Based Skin Cancer Detection and Classification Using Dermoscopic Image Analysis

**Abstract.** An innovative artificial intelligence framework is introduced for automated skin cancer detection using deep learning architectures and computer vision techniques. Raw dermoscopic images undergo image enhancement, resizing, and pixel normalization. Convolutional neural networks extract key diagnostic features such as lesion shape, color variations, and border irregularities. Transfer learning models classify lesions into benign or malignant categories with confidence scores. Explainable AI methods like Grad-CAM heat maps highlight the regions influencing the model's decisions. The system generates clinical evaluation metrics and exports reports as PDF or CSV.

This notebook runs the complete pipeline end-to-end on **Google Colab** (Runtime → Change runtime type → **GPU**):

1. Download HAM10000 from Kaggle  2. Exploratory data analysis  3. Preprocessing (hair removal, CLAHE)
4. Leakage-free split by `lesion_id`  5. Transfer-learning training  6. Evaluation  7. Grad-CAM  8. Model comparison

In [ ]:
!nvidia-smi || echo "No NVIDIA GPU - switch the Colab runtime to GPU for reasonable training times"
!pip install -q -U kaggle reportlab
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Project source files
The cells below write the project modules to disk (identical to the files in the repository).

In [ ]:
import os
for d in ["scripts", "src", "data/raw", "models", "results", "sample_images", "docs"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile config.py
"""Central configuration: paths, hyperparameters, class names and label mappings.

All paths use pathlib so the project runs unchanged on macOS, Linux and Windows.
"""
from pathlib import Path

# --------------------------------------------------------------------------- paths
ROOT_DIR = Path(__file__).resolve().parent
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = ROOT_DIR / "models"
RESULTS_DIR = ROOT_DIR / "results"
SAMPLE_DIR = ROOT_DIR / "sample_images"
DOCS_DIR = ROOT_DIR / "docs"

METADATA_CSV_NAME = "HAM10000_metadata.csv"
KAGGLE_DATASET = "kmader/skin-cancer-mnist-ham10000"

# --------------------------------------------------------------------------- classes
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_FULL_NAMES = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}
MALIGNANT_CLASSES = ["mel", "bcc", "akiec"]
BENIGN_CLASSES = ["nv", "bkl", "df", "vasc"]
BINARY_MAP = {c: ("Malignant" if c in MALIGNANT_CLASSES else "Benign") for c in CLASS_NAMES}
NUM_CLASSES = len(CLASS_NAMES)

# --------------------------------------------------------------------------- preprocessing
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
HAIR_KERNEL_SIZE = 17          # blackhat structuring element size
HAIR_THRESHOLD = 12            # threshold on blackhat response to build the hair mask
HAIR_MIN_LENGTH = 40           # keep only long, thin mask components (hairs), not skin texture
INPAINT_RADIUS = 3
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)

# --------------------------------------------------------------------------- splits
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
SEED = 42

# --------------------------------------------------------------------------- training
AVAILABLE_MODELS = ["efficientnet_b0", "resnet50", "densenet121"]
DEFAULT_MODEL = "efficientnet_b0"
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 4
MIN_EPOCHS = 10                # early stopping can only trigger after this many epochs
NUM_WORKERS = 0                # 0 avoids multiprocessing issues on Windows/macOS
QUICK_SAMPLES_PER_CLASS = 40   # --quick mode subset size
QUICK_EPOCHS = 2


def get_device():
    """CUDA if available, else Apple MPS, else CPU."""
    import torch

    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def model_path(model_name: str, quick: bool = False) -> Path:
    """Checkpoint location, shared by src/train.py and the training notebooks."""
    return MODELS_DIR / f"{model_name}{'_quick' if quick else ''}.pth"


for _d in (RAW_DIR, PROCESSED_DIR, SPLITS_DIR, MODELS_DIR, RESULTS_DIR, SAMPLE_DIR, DOCS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile scripts/download_data.py
"""Download the HAM10000 dataset from Kaggle into data/raw/.

Supports both Kaggle credential formats:
  * new API token:  ~/.kaggle/access_token  (or KAGGLE_API_TOKEN env var)
  * legacy:         ~/.kaggle/kaggle.json   (or KAGGLE_USERNAME / KAGGLE_KEY env vars)

Usage:
    python scripts/download_data.py
"""
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402

MANUAL_INSTRUCTIONS = f"""
------------------------------------------------------------------------------
Could not download HAM10000 automatically.

Option A - Kaggle API credentials
  1. Sign in at https://www.kaggle.com -> Settings -> API -> "Generate New Token".
  2. Save the token:
       macOS/Linux : ~/.kaggle/access_token        (new format, token string only)
                     or ~/.kaggle/kaggle.json      (legacy format)
       Windows     : %USERPROFILE%\\.kaggle\\access_token  (or kaggle.json)
     Alternatively set the KAGGLE_API_TOKEN environment variable.
  3. Re-run:  python scripts/download_data.py

Option B - Manual download
  1. Open https://www.kaggle.com/datasets/{config.KAGGLE_DATASET}
  2. Click "Download", unzip the archive into:
       {config.RAW_DIR}
  3. You should end up with {config.METADATA_CSV_NAME} and the folders
     HAM10000_images_part_1/ and HAM10000_images_part_2/ inside data/raw/.
------------------------------------------------------------------------------
"""


def has_credentials() -> bool:
    kaggle_dir = Path(os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle"))
    return any([
        os.environ.get("KAGGLE_API_TOKEN"),
        os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"),
        (kaggle_dir / "access_token").exists(),
        (kaggle_dir / "kaggle.json").exists(),
    ])


def dataset_present() -> bool:
    return any(config.RAW_DIR.rglob(config.METADATA_CSV_NAME)) and \
        sum(1 for _ in config.RAW_DIR.rglob("ISIC_*.jpg")) >= 10000


def main() -> int:
    if dataset_present():
        print(f"HAM10000 already present in {config.RAW_DIR} - nothing to do.")
        return 0

    if not has_credentials():
        print("No Kaggle credentials found (access_token, kaggle.json or env vars).")
        print(MANUAL_INSTRUCTIONS)
        return 1

    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError:
        print("The 'kaggle' package is not installed. Run: pip install -U kaggle")
        print(MANUAL_INSTRUCTIONS)
        return 1

    try:
        api = KaggleApi()
        api.authenticate()
        print(f"Downloading {config.KAGGLE_DATASET} into {config.RAW_DIR} (~2.7 GB) ...")
        api.dataset_download_files(config.KAGGLE_DATASET, path=str(config.RAW_DIR),
                                   unzip=True, quiet=False)
    except Exception as exc:  # network / auth / quota errors
        print(f"Kaggle download failed: {exc}")
        print(MANUAL_INSTRUCTIONS)
        return 1

    if not dataset_present():
        print("Download finished but the expected files were not found.")
        print(MANUAL_INSTRUCTIONS)
        return 1
    n_imgs = sum(1 for _ in config.RAW_DIR.rglob("ISIC_*.jpg"))
    print(f"Done. Found {n_imgs} image files and metadata in {config.RAW_DIR}.")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/prepare_samples.py
"""Copy a handful of held-out TEST images (different classes) into sample_images/ for the demo.

If results/<model>/test_predictions.csv exists, one correctly classified image per class is chosen
(median confidence among correct predictions, so the demo is representative rather than cherry-picked
for maximum confidence). Otherwise a random test image per class is used.

Usage:
    python scripts/prepare_samples.py
"""
import shutil
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402

SAMPLE_CLASSES = ["mel", "bcc", "akiec", "nv", "bkl", "vasc"]


def main(model_name: str = config.DEFAULT_MODEL):
    test = pd.read_csv(config.SPLITS_DIR / "test.csv")
    preds_file = config.RESULTS_DIR / model_name / "test_predictions.csv"
    preds = pd.read_csv(preds_file) if preds_file.exists() else None

    for old in config.SAMPLE_DIR.glob("*.jpg"):
        old.unlink()
    for cls in SAMPLE_CLASSES:
        pick = None
        if preds is not None:
            ok = preds[(preds.dx == cls) & (preds.pred == cls)].sort_values("confidence")
            if len(ok):
                pick = ok.iloc[len(ok) // 2]["image_id"]
        if pick is None:
            pick = test[test.dx == cls].sample(1, random_state=config.SEED)["image_id"].iloc[0]
        src = config.RAW_DIR / Path(test.loc[test.image_id == pick, "path"].iloc[0])
        dst = config.SAMPLE_DIR / f"{cls}_{pick}.jpg"
        shutil.copy2(src, dst)
        print(f"{cls:6s} -> {dst.name}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/preprocess.py
"""Dermoscopic image preprocessing.

Pipeline (identical for training and inference):
    1. Hair removal   - morphological black-hat on the grey image -> hair mask -> Telea inpainting
    2. Enhancement    - CLAHE on the L channel of the LAB colour space
    3. Resize         - 224 x 224 (INTER_AREA)
    4. Normalisation  - scale to [0, 1] then ImageNet mean / std (done in the torch transforms)

Run as a script to cache the preprocessed dataset to data/processed/ (speeds up training):
    python src/preprocess.py
"""
import sys
from pathlib import Path

import cv2
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402


def load_image_rgb(path) -> np.ndarray:
    """Read an image from disk as RGB uint8 (works with non-ASCII Windows paths)."""
    data = np.fromfile(str(path), dtype=np.uint8)
    bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def hair_mask(img_rgb: np.ndarray,
              kernel_size: int = config.HAIR_KERNEL_SIZE,
              threshold: int = config.HAIR_THRESHOLD,
              min_length: int = config.HAIR_MIN_LENGTH):
    """Black-hat response and binary hair mask.

    The black-hat transform responds to thin dark structures. Thresholding alone also catches skin
    texture and the lesion's pigment network, so only connected components that are long
    (bounding box >= min_length px) and sparse (fill ratio < 0.4), i.e. hair-shaped, are kept.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    w, h, area = stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT], stats[:, cv2.CC_STAT_AREA]
    keep = (np.maximum(w, h) >= min_length) & (area / np.maximum(w * h, 1) < 0.4)
    keep[0] = False  # background
    mask = np.where(keep[labels], 255, 0).astype(np.uint8)
    return blackhat, cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)


def remove_hair(img_rgb: np.ndarray, radius: int = config.INPAINT_RADIUS) -> np.ndarray:
    """Remove dark hair strands: black-hat hair mask followed by Telea inpainting."""
    _, mask = hair_mask(img_rgb)
    return cv2.inpaint(img_rgb, mask, radius, cv2.INPAINT_TELEA)


def apply_clahe(img_rgb: np.ndarray,
                clip_limit: float = config.CLAHE_CLIP_LIMIT,
                tile_grid=config.CLAHE_TILE_GRID) -> np.ndarray:
    """Contrast Limited Adaptive Histogram Equalisation on the luminance channel."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l_chan, a_chan, b_chan = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tuple(tile_grid))
    lab = cv2.merge((clahe.apply(l_chan), a_chan, b_chan))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def resize(img_rgb: np.ndarray, size: int = config.IMG_SIZE) -> np.ndarray:
    return cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_AREA)


def normalize(img_rgb: np.ndarray) -> np.ndarray:
    """uint8 HxWx3 -> float32 HxWx3 normalised with ImageNet statistics."""
    img = img_rgb.astype(np.float32) / 255.0
    return (img - np.array(config.IMAGENET_MEAN, np.float32)) / np.array(config.IMAGENET_STD, np.float32)


def preprocess_image(img_rgb: np.ndarray, size: int = config.IMG_SIZE) -> np.ndarray:
    """Hair removal -> CLAHE -> resize. Returns an RGB uint8 image (size x size).

    Hair removal runs at a fixed working resolution so the morphological kernel
    behaves the same regardless of the input image size.
    """
    h, w = img_rgb.shape[:2]
    scale = 600.0 / max(h, w)
    if abs(scale - 1.0) > 0.05:
        img_rgb = cv2.resize(img_rgb, (int(round(w * scale)), int(round(h * scale))),
                             interpolation=cv2.INTER_AREA)
    img = remove_hair(img_rgb)
    img = apply_clahe(img)
    return resize(img, size)


def preprocess_path(path, size: int = config.IMG_SIZE) -> np.ndarray:
    return preprocess_image(load_image_rgb(path), size)


def save_image_rgb(img_rgb: np.ndarray, path) -> None:
    ok, buf = cv2.imencode(Path(path).suffix or ".jpg", cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR),
                           [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        raise ValueError(f"Could not encode image for {path}")
    buf.tofile(str(path))


def cache_dataset(force: bool = False) -> int:
    """Preprocess every HAM10000 image once into data/processed/<image_id>.jpg."""
    from tqdm import tqdm
    from dataset import load_metadata

    df = load_metadata()
    done = 0
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Preprocessing"):
        out = config.PROCESSED_DIR / f"{row.image_id}.jpg"
        if out.exists() and not force:
            continue
        save_image_rgb(preprocess_path(row.path), out)
        done += 1
    print(f"Preprocessed {done} new images into {config.PROCESSED_DIR}")
    return done


if __name__ == "__main__":
    import argparse

    sys.path.insert(0, str(Path(__file__).resolve().parent))
    parser = argparse.ArgumentParser(description="Cache preprocessed HAM10000 images")
    parser.add_argument("--force", action="store_true", help="re-process existing files")
    cache_dataset(parser.parse_args().force)

In [ ]:
%%writefile src/dataset.py
"""HAM10000 metadata loading, leakage-free splitting and the PyTorch Dataset.

Splits are stratified by diagnosis and grouped by lesion_id: every image of a given
lesion lands in exactly one of train / val / test (70 / 15 / 15).
"""
import sys
from pathlib import Path

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from preprocess import load_image_rgb, preprocess_path  # noqa: E402

CLASS_TO_IDX = {c: i for i, c in enumerate(config.CLASS_NAMES)}


# --------------------------------------------------------------------------- metadata
def find_metadata_csv() -> Path:
    matches = sorted(config.RAW_DIR.rglob(config.METADATA_CSV_NAME))
    if not matches:
        raise FileNotFoundError(
            f"{config.METADATA_CSV_NAME} not found under {config.RAW_DIR}. "
            "Run: python scripts/download_data.py")
    return matches[0]


def load_metadata() -> pd.DataFrame:
    """Metadata with an absolute `path` column and integer `label`."""
    df = pd.read_csv(find_metadata_csv())
    paths = {}
    for p in config.RAW_DIR.rglob("ISIC_*.jpg"):
        paths.setdefault(p.stem, p)  # the Kaggle archive contains duplicate folders
    df["path"] = df["image_id"].map(paths)
    missing = df["path"].isna().sum()
    if missing:
        print(f"Warning: {missing} images listed in metadata were not found; dropping them.")
        df = df.dropna(subset=["path"])
    df["label"] = df["dx"].map(CLASS_TO_IDX)
    df["binary"] = df["dx"].map(config.BINARY_MAP)
    return df.reset_index(drop=True)


def _resolve_paths(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["path"] = df["path"].map(lambda rel: str(config.RAW_DIR / Path(rel)))
    return df


def make_splits(force: bool = False) -> dict:
    """Create (or load cached) train/val/test splits grouped by lesion_id."""
    files = {s: config.SPLITS_DIR / f"{s}.csv" for s in ("train", "val", "test")}
    if not force and all(f.exists() for f in files.values()):
        return {s: _resolve_paths(pd.read_csv(f)) for s, f in files.items()}

    df = load_metadata()
    lesions = df.groupby("lesion_id")["dx"].first().reset_index()
    train_l, rest_l = train_test_split(
        lesions, test_size=config.VAL_FRAC + config.TEST_FRAC,
        stratify=lesions["dx"], random_state=config.SEED)
    val_l, test_l = train_test_split(
        rest_l, test_size=config.TEST_FRAC / (config.VAL_FRAC + config.TEST_FRAC),
        stratify=rest_l["dx"], random_state=config.SEED)

    splits = {}
    for name, lesion_df in (("train", train_l), ("val", val_l), ("test", test_l)):
        part = df[df["lesion_id"].isin(set(lesion_df["lesion_id"]))].copy()
        # store paths relative to data/raw (POSIX style) so the CSVs are portable across machines/OSes
        part["path"] = part["path"].map(lambda p: Path(p).relative_to(config.RAW_DIR).as_posix())
        part.to_csv(files[name], index=False)
        splits[name] = _resolve_paths(part)
    assert not (set(splits["train"].lesion_id) & set(splits["test"].lesion_id)), "lesion leakage"
    assert not (set(splits["train"].lesion_id) & set(splits["val"].lesion_id)), "lesion leakage"
    for name, part in splits.items():
        print(f"{name:5s}: {len(part):5d} images, {part.lesion_id.nunique():5d} lesions")
    return splits


def quick_subset(df: pd.DataFrame, per_class: int) -> pd.DataFrame:
    shuffled = df.sample(frac=1.0, random_state=config.SEED)
    return shuffled.groupby("dx").head(per_class).reset_index(drop=True)


# --------------------------------------------------------------------------- transforms
def get_transforms(train: bool):
    norm = [transforms.ToTensor(),
            transforms.Normalize(config.IMAGENET_MEAN, config.IMAGENET_STD)]
    if not train:
        return transforms.Compose([transforms.ToPILImage()] + norm)
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),  # zoom
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03),
    ] + norm)


# --------------------------------------------------------------------------- dataset
class SkinLesionDataset(Dataset):
    """Returns (normalised tensor, label). Uses cached preprocessed images when present."""

    def __init__(self, df: pd.DataFrame, train: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = get_transforms(train)

    def __len__(self):
        return len(self.df)

    def load_preprocessed(self, idx: int):
        row = self.df.iloc[idx]
        cached = config.PROCESSED_DIR / f"{row.image_id}.jpg"
        if cached.exists():
            return load_image_rgb(cached)
        return preprocess_path(Path(row.path))

    def __getitem__(self, idx):
        img = self.load_preprocessed(idx)
        return self.transform(img), int(self.df.iloc[idx]["label"])


def get_dataloaders(batch_size: int = config.BATCH_SIZE, quick: bool = False):
    splits = make_splits()
    train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]
    if quick:
        train_df = quick_subset(train_df, config.QUICK_SAMPLES_PER_CLASS)
        val_df = quick_subset(val_df, config.QUICK_SAMPLES_PER_CLASS // 2)
        test_df = quick_subset(test_df, config.QUICK_SAMPLES_PER_CLASS // 2)
    pin = torch.cuda.is_available()
    loaders = {
        "train": DataLoader(SkinLesionDataset(train_df, train=True), batch_size=batch_size,
                            shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=pin),
        "val": DataLoader(SkinLesionDataset(val_df), batch_size=batch_size,
                          shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=pin),
        "test": DataLoader(SkinLesionDataset(test_df), batch_size=batch_size,
                           shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=pin),
    }
    return loaders, {"train": train_df, "val": val_df, "test": test_df}


if __name__ == "__main__":
    s = make_splits(force=True)
    for k, v in s.items():
        print(k, v["dx"].value_counts().to_dict())

In [ ]:
%%writefile src/models.py
"""Transfer-learning backbones (ImageNet pre-trained) with a new 7-class head."""
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402

DISPLAY_NAMES = {
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet50",
    "densenet121": "DenseNet121",
}


def build_model(name: str, num_classes: int = config.NUM_CLASSES, pretrained: bool = True) -> nn.Module:
    if name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        in_f = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    elif name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier.in_features, num_classes))
    else:
        raise ValueError(f"Unknown model '{name}'. Choose from {config.AVAILABLE_MODELS}")
    return model


def get_target_layer(model: nn.Module, name: str) -> nn.Module:
    """Last convolutional block, used as the Grad-CAM target."""
    if name == "efficientnet_b0":
        return model.features[-1]
    if name == "resnet50":
        return model.layer4[-1]
    if name == "densenet121":
        # denseblock4 output (norm5 is followed by an in-place ReLU, which breaks tensor hooks)
        return model.features.denseblock4
    raise ValueError(name)


def load_checkpoint(path, device=None):
    """Load a checkpoint saved by train.py. Returns (model, checkpoint_dict)."""
    device = device or config.get_device()
    ckpt = torch.load(str(path), map_location="cpu", weights_only=False)
    model = build_model(ckpt["model_name"], len(ckpt["class_names"]), pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    return model, ckpt


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
%%writefile src/train.py
"""Train a transfer-learning classifier on HAM10000.

Examples:
    python src/train.py --quick                       # 2-epoch smoke test on a small subset
    python src/train.py --model efficientnet_b0 --epochs 15
    python src/train.py --model resnet50 --epochs 10 --batch-size 32
"""
import argparse
import json
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
from sklearn.metrics import f1_score  # noqa: E402
from sklearn.utils.class_weight import compute_class_weight  # noqa: E402
from tqdm import tqdm  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from dataset import get_dataloaders  # noqa: E402
from models import build_model  # noqa: E402


def set_seed(seed: int = config.SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def class_weights(train_df: pd.DataFrame, device) -> torch.Tensor:
    """Balanced class weights (n_samples / (n_classes * n_c)), square-rooted to avoid over-correction."""
    labels = train_df["label"].to_numpy()
    present = np.unique(labels)
    w = np.ones(config.NUM_CLASSES, dtype=np.float32)
    w[present] = compute_class_weight("balanced", classes=present, y=labels)
    w = np.sqrt(w)
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)


def run_epoch(model, loader, criterion, device, optimizer=None, desc=""):
    train = optimizer is not None
    model.train(train)
    total_loss, preds, targets = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, desc=desc, leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds.append(logits.argmax(1).cpu())
            targets.append(y.cpu())
    preds, targets = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return {
        "loss": total_loss / len(loader.dataset),
        "acc": float((preds == targets).mean()),
        "f1": float(f1_score(targets, preds, average="macro", zero_division=0)),
    }


def plot_history(hist: pd.DataFrame, out: Path, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, label in zip(axes, ["loss", "acc", "f1"], ["Loss", "Accuracy", "Macro F1"]):
        ax.plot(hist["epoch"], hist[f"train_{key}"], "o-", label="train")
        ax.plot(hist["epoch"], hist[f"val_{key}"], "s-", label="validation")
        ax.set_xlabel("Epoch")
        ax.set_title(label)
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


def train(model_name: str = config.DEFAULT_MODEL, epochs: int = config.EPOCHS,
          batch_size: int = config.BATCH_SIZE, lr: float = config.LEARNING_RATE,
          quick: bool = False, patience: int = config.EARLY_STOPPING_PATIENCE) -> Path:
    set_seed()
    device = config.get_device()
    if quick:
        epochs = min(epochs, config.QUICK_EPOCHS)
    print(f"Device: {device} | model: {model_name} | epochs: {epochs} | batch: {batch_size} | quick: {quick}")

    loaders, dfs = get_dataloaders(batch_size, quick=quick)
    print(f"train={len(dfs['train'])} val={len(dfs['val'])} test={len(dfs['test'])}")

    model = build_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights(dfs["train"], device), label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=2)

    suffix = "_quick" if quick else ""
    ckpt_path = config.model_path(model_name, quick)
    out_dir = config.RESULTS_DIR / f"{model_name}{suffix}"
    out_dir.mkdir(parents=True, exist_ok=True)

    best_f1, bad_epochs, history = -1.0, 0, []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        te = time.time()
        tr = run_epoch(model, loaders["train"], criterion, device, optimizer, f"train {epoch}/{epochs}")
        va = run_epoch(model, loaders["val"], criterion, device, None, f"val   {epoch}/{epochs}")
        scheduler.step(va["f1"])
        lr_now = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()},
                        **{f"val_{k}": v for k, v in va.items()}, "lr": lr_now,
                        "seconds": time.time() - te})
        print(f"Epoch {epoch:2d}: train loss {tr['loss']:.4f} acc {tr['acc']:.4f} f1 {tr['f1']:.4f} | "
              f"val loss {va['loss']:.4f} acc {va['acc']:.4f} f1 {va['f1']:.4f} | "
              f"lr {lr_now:.1e} | {time.time() - te:.0f}s")

        if va["f1"] > best_f1:
            best_f1, bad_epochs = va["f1"], 0
            torch.save({
                "model_name": model_name,
                "state_dict": model.state_dict(),
                "class_names": config.CLASS_NAMES,
                "img_size": config.IMG_SIZE,
                "epoch": epoch,
                "val_f1": va["f1"],
                "val_acc": va["acc"],
                "quick": quick,
            }, ckpt_path)
            print(f"  -> saved best model (val macro-F1 {best_f1:.4f}) to {ckpt_path.name}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience and (quick or epoch >= config.MIN_EPOCHS):
                print(f"Early stopping after {epoch} epochs (no improvement for {patience}).")
                break

    hist = pd.DataFrame(history)
    hist.to_csv(out_dir / "training_history.csv", index=False)
    plot_history(hist, out_dir / "training_curves.png", f"Training curves - {model_name}")
    with open(out_dir / "training_summary.json", "w") as f:
        json.dump({"model": model_name, "epochs_run": len(history), "best_val_f1": best_f1,
                   "train_minutes": (time.time() - t0) / 60, "device": str(device),
                   "train_size": len(dfs["train"]), "quick": quick}, f, indent=2)
    print(f"Training done in {(time.time() - t0) / 60:.1f} min. Best val macro-F1: {best_f1:.4f}")
    return ckpt_path


def parse_args():
    p = argparse.ArgumentParser(description="Train a skin lesion classifier")
    p.add_argument("--model", default=config.DEFAULT_MODEL, choices=config.AVAILABLE_MODELS)
    p.add_argument("--epochs", type=int, default=config.EPOCHS)
    p.add_argument("--batch-size", type=int, default=config.BATCH_SIZE)
    p.add_argument("--lr", type=float, default=config.LEARNING_RATE)
    p.add_argument("--patience", type=int, default=config.EARLY_STOPPING_PATIENCE)
    p.add_argument("--quick", action="store_true", help="small subset, 2 epochs (pipeline check)")
    return p.parse_args()


if __name__ == "__main__":
    a = parse_args()
    train(a.model, a.epochs, a.batch_size, a.lr, a.quick, a.patience)

In [ ]:
%%writefile src/evaluate.py
"""Evaluate a trained checkpoint on the held-out test split.

Outputs (results/<model>/):
    metrics.json, per_class_metrics.csv, classification_report.txt,
    confusion_matrix.png, confusion_matrix_binary.png, roc_curves.png, test_predictions.csv

Usage:
    python src/evaluate.py --model efficientnet_b0
    python src/evaluate.py --model efficientnet_b0 --quick
"""
import argparse
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402
import torch  # noqa: E402
from sklearn.metrics import (accuracy_score, auc, classification_report, confusion_matrix,  # noqa: E402
                             f1_score, precision_recall_fscore_support, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize  # noqa: E402
from tqdm import tqdm  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from dataset import get_dataloaders  # noqa: E402
from models import load_checkpoint  # noqa: E402

MALIGNANT_IDX = [config.CLASS_NAMES.index(c) for c in config.MALIGNANT_CLASSES]


@torch.no_grad()
def predict_loader(model, loader, device):
    probs, labels = [], []
    for x, y in tqdm(loader, desc="Evaluating", leave=False):
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def specificity_per_class(cm: np.ndarray) -> np.ndarray:
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    tn = cm.sum() - tp - fp - fn
    return tn / np.maximum(tn + fp, 1)


def compute_metrics(probs: np.ndarray, labels: np.ndarray) -> dict:
    names = config.CLASS_NAMES
    preds = probs.argmax(1)
    k = len(names)
    cm = confusion_matrix(labels, preds, labels=range(k))
    prec, rec, f1, support = precision_recall_fscore_support(labels, preds, labels=range(k), zero_division=0)
    spec = specificity_per_class(cm)
    y_bin = label_binarize(labels, classes=range(k))
    aucs = []
    for i in range(k):
        aucs.append(roc_auc_score(y_bin[:, i], probs[:, i]) if 0 < y_bin[:, i].sum() < len(labels) else float("nan"))

    per_class = pd.DataFrame({
        "class": names, "full_name": [config.CLASS_FULL_NAMES[c] for c in names],
        "type": [config.BINARY_MAP[c] for c in names],
        "precision": prec, "recall_sensitivity": rec, "specificity": spec,
        "f1": f1, "roc_auc": aucs, "support": support})

    # benign vs malignant: class prediction mapped to its group; score = summed malignant prob
    true_m = np.isin(labels, MALIGNANT_IDX).astype(int)
    pred_m = np.isin(preds, MALIGNANT_IDX).astype(int)
    score_m = probs[:, MALIGNANT_IDX].sum(1)
    bcm = confusion_matrix(true_m, pred_m, labels=[0, 1])
    tn, fp, fn, tp = bcm.ravel()
    binary = {
        "accuracy": float((tp + tn) / bcm.sum()),
        "precision": float(tp / max(tp + fp, 1)),
        "sensitivity": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "f1": float(f1_score(true_m, pred_m, zero_division=0)),
        "roc_auc": float(roc_auc_score(true_m, score_m)) if 0 < true_m.sum() < len(true_m) else float("nan"),
        "confusion_matrix": bcm.tolist(),
    }
    overall = {
        "accuracy": float(accuracy_score(labels, preds)),
        "balanced_accuracy": float(np.mean(rec)),
        "macro_precision": float(np.mean(prec)),
        "macro_recall_sensitivity": float(np.mean(rec)),
        "macro_specificity": float(np.mean(spec)),
        "macro_f1": float(np.mean(f1)),
        "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
        "macro_roc_auc": float(np.nanmean(aucs)),
        "n_test": int(len(labels)),
    }
    return {"overall": overall, "per_class": per_class, "binary": binary, "confusion_matrix": cm}


def plot_confusion(cm, names, out, title, normalize=True):
    data = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1) if normalize else cm
    annot = np.array([[f"{cm[i, j]}\n({data[i, j]:.0%})" for j in range(cm.shape[1])] for i in range(cm.shape[0])])
    fig, ax = plt.subplots(figsize=(1.1 * len(names) + 2.5, 1.0 * len(names) + 2))
    sns.heatmap(data, annot=annot, fmt="", cmap="Blues", xticklabels=names, yticklabels=names,
                vmin=0, vmax=1, cbar_kws={"label": "Row-normalised rate"}, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


def plot_roc(probs, labels, out, title):
    names = config.CLASS_NAMES
    y_bin = label_binarize(labels, classes=range(len(names)))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for i, c in enumerate(names):
        if 0 < y_bin[:, i].sum() < len(labels):
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
            axes[0].plot(fpr, tpr, lw=1.6, label=f"{c} (AUC={auc(fpr, tpr):.3f})")
    axes[0].set_title(f"{title}: one-vs-rest ROC (7 classes)")
    true_m = np.isin(labels, MALIGNANT_IDX).astype(int)
    if 0 < true_m.sum() < len(true_m):
        fpr, tpr, _ = roc_curve(true_m, probs[:, MALIGNANT_IDX].sum(1))
        axes[1].plot(fpr, tpr, lw=2.2, color="#c0392b", label=f"Malignant vs benign (AUC={auc(fpr, tpr):.3f})")
    axes[1].set_title(f"{title}: benign vs malignant ROC")
    for ax in axes:
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
        ax.set_xlabel("False positive rate (1 - specificity)")
        ax.set_ylabel("True positive rate (sensitivity)")
        ax.legend(loc="lower right", fontsize=9)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


def evaluate(model_name: str = config.DEFAULT_MODEL, quick: bool = False, batch_size: int = config.BATCH_SIZE) -> dict:
    suffix = "_quick" if quick else ""
    ckpt_path = config.model_path(model_name, quick)
    out_dir = config.RESULTS_DIR / f"{model_name}{suffix}"
    out_dir.mkdir(parents=True, exist_ok=True)
    device = config.get_device()
    model, ckpt = load_checkpoint(ckpt_path, device)
    loaders, dfs = get_dataloaders(batch_size, quick=quick)
    probs, labels = predict_loader(model, loaders["test"], device)
    m = compute_metrics(probs, labels)

    names = config.CLASS_NAMES
    m["per_class"].to_csv(out_dir / "per_class_metrics.csv", index=False)
    report = classification_report(labels, probs.argmax(1), labels=range(len(names)),
                                   target_names=names, digits=4, zero_division=0)
    (out_dir / "classification_report.txt").write_text(report)
    plot_confusion(m["confusion_matrix"], names, out_dir / "confusion_matrix.png",
                   f"Confusion matrix - {model_name} (test set)")
    plot_confusion(np.array(m["binary"]["confusion_matrix"]), ["Benign", "Malignant"],
                   out_dir / "confusion_matrix_binary.png", f"Benign vs malignant - {model_name}")
    plot_roc(probs, labels, out_dir / "roc_curves.png", model_name)

    pred_df = dfs["test"][["image_id", "lesion_id", "dx"]].copy().reset_index(drop=True)
    pred_df["pred"] = [names[i] for i in probs.argmax(1)]
    pred_df["confidence"] = probs.max(1)
    for i, c in enumerate(names):
        pred_df[f"p_{c}"] = probs[:, i]
    pred_df.to_csv(out_dir / "test_predictions.csv", index=False)

    summary = {"model": model_name, "checkpoint": ckpt_path.name, "best_epoch": ckpt.get("epoch"),
               "overall": m["overall"], "binary": m["binary"],
               "per_class": m["per_class"].drop(columns=["full_name"]).to_dict(orient="records")}
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(report)
    print("Overall:", json.dumps(m["overall"], indent=2))
    print("Benign vs malignant:", json.dumps({k: v for k, v in m["binary"].items() if k != "confusion_matrix"}, indent=2))
    print(f"Saved results to {out_dir}")
    return summary


if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Evaluate a trained model on the test split")
    p.add_argument("--model", default=config.DEFAULT_MODEL, choices=config.AVAILABLE_MODELS)
    p.add_argument("--batch-size", type=int, default=config.BATCH_SIZE)
    p.add_argument("--quick", action="store_true")
    a = p.parse_args()
    evaluate(a.model, a.quick, a.batch_size)

In [ ]:
%%writefile src/gradcam.py
"""Grad-CAM (Selvaraju et al., 2017) implemented from scratch with PyTorch hooks.

    alpha_k  = mean_{i,j} dy_c / dA^k_{ij}            (channel importance weights)
    L_c      = ReLU( sum_k alpha_k * A^k )             (class activation map)

Usage:
    python src/gradcam.py --image sample_images/mel_ISIC_0000000.jpg
    python src/gradcam.py --all-samples              # every image in sample_images/
"""
import argparse
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402


class GradCAM:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        self._handle = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach()
        # tensor hook: called with dL/d(output) during backward
        output.register_hook(self._save_grad)

    def _save_grad(self, grad):
        self.gradients = grad.detach()

    def __call__(self, x: torch.Tensor, class_idx: int = None):
        """x: (1,3,H,W). Returns (cam HxW in [0,1], class_idx, probs)."""
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = self.model(x)
            probs = torch.softmax(logits, 1)[0].detach().cpu().numpy()
            if class_idx is None:
                class_idx = int(logits.argmax(1).item())
            logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)            # (1,K,1,1)
        cam = F.relu((weights * self.activations).sum(1, keepdim=True))    # (1,1,h,w)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, probs

    def remove(self):
        self._handle.remove()


def overlay_heatmap(img_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Blend a JET-coloured CAM over an RGB uint8 image."""
    cam = cv2.resize(cam, (img_rgb.shape[1], img_rgb.shape[0]))
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return np.uint8(np.clip((1 - alpha) * img_rgb + alpha * heat, 0, 255))


def save_figure(original, preprocessed, overlay, title, out_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
    for ax, im, t in zip(axes, [original, preprocessed, overlay],
                         ["Original", "Preprocessed (hair removal + CLAHE)", "Grad-CAM overlay"]):
        ax.imshow(im)
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)


if __name__ == "__main__":
    from predict import Predictor

    p = argparse.ArgumentParser(description="Generate Grad-CAM visualisations")
    p.add_argument("--image", type=Path, help="path to a dermoscopic image")
    p.add_argument("--all-samples", action="store_true", help="run on every file in sample_images/")
    p.add_argument("--model", default=config.DEFAULT_MODEL, choices=config.AVAILABLE_MODELS)
    p.add_argument("--out-dir", type=Path, default=config.RESULTS_DIR / "gradcam")
    a = p.parse_args()

    images = sorted(config.SAMPLE_DIR.glob("*.jpg")) if a.all_samples else [a.image]
    if not images or images[0] is None:
        p.error("give --image PATH or --all-samples")
    a.out_dir.mkdir(parents=True, exist_ok=True)
    predictor = Predictor(model_name=a.model)
    for img_path in images:
        r = predictor.predict(img_path)
        title = (f"{img_path.name}: predicted {r['pred_class']} ({r['confidence']:.1%}) - "
                 f"{r['binary_label']}")
        out = a.out_dir / f"gradcam_{img_path.stem}.png"
        save_figure(r["original"], r["preprocessed"], r["overlay"], title, out)
        print(f"{title}  ->  {out}")

In [ ]:
%%writefile src/predict.py
"""Single-image inference with Grad-CAM.

Usage:
    python src/predict.py --image sample_images/<file>.jpg
    python src/predict.py --image path/to/img.jpg --model resnet50 --pdf --csv
"""
import argparse
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from gradcam import GradCAM, overlay_heatmap  # noqa: E402
from models import DISPLAY_NAMES, get_target_layer, load_checkpoint  # noqa: E402
from preprocess import load_image_rgb, normalize, preprocess_image  # noqa: E402


def find_checkpoint(model_name: str = config.DEFAULT_MODEL) -> Path:
    """Prefer the fully trained checkpoint, fall back to the --quick one."""
    for p in (config.model_path(model_name), config.model_path(model_name, quick=True)):
        if p.exists():
            return p
    raise FileNotFoundError(f"No checkpoint for {model_name} in {config.MODELS_DIR}. "
                            f"Train first: python src/train.py --model {model_name}")


class Predictor:
    def __init__(self, model_name: str = config.DEFAULT_MODEL, checkpoint: Path = None, device=None):
        self.device = device or config.get_device()
        self.checkpoint = Path(checkpoint) if checkpoint else find_checkpoint(model_name)
        self.model, self.ckpt = load_checkpoint(self.checkpoint, self.device)
        self.model_name = self.ckpt["model_name"]
        self.display_name = DISPLAY_NAMES.get(self.model_name, self.model_name)
        self.class_names = self.ckpt["class_names"]
        self.cam = GradCAM(self.model, get_target_layer(self.model, self.model_name))

    def predict_array(self, img_rgb: np.ndarray) -> dict:
        pre = preprocess_image(img_rgb)
        x = torch.from_numpy(normalize(pre).transpose(2, 0, 1)).unsqueeze(0).to(self.device)
        cam, idx, probs = self.cam(x)
        cls = self.class_names[idx]
        malignant_prob = float(sum(probs[self.class_names.index(c)] for c in config.MALIGNANT_CLASSES))
        return {
            "pred_class": cls,
            "pred_full_name": config.CLASS_FULL_NAMES[cls],
            "binary_label": config.BINARY_MAP[cls],
            "confidence": float(probs[idx]),
            "malignant_probability": malignant_prob,
            "probabilities": {c: float(p) for c, p in zip(self.class_names, probs)},
            "original": img_rgb,
            "preprocessed": pre,
            "cam": cam,
            "overlay": overlay_heatmap(pre, cam),
            "model": self.display_name,
        }

    def predict(self, path) -> dict:
        r = self.predict_array(load_image_rgb(path))
        r["filename"] = Path(path).name
        return r


def main():
    p = argparse.ArgumentParser(description="Classify a single dermoscopic image")
    p.add_argument("--image", type=Path, required=True)
    p.add_argument("--model", default=config.DEFAULT_MODEL, choices=config.AVAILABLE_MODELS)
    p.add_argument("--pdf", action="store_true", help="also write a PDF report to results/reports/")
    p.add_argument("--csv", action="store_true", help="also append a row to results/reports/predictions.csv")
    a = p.parse_args()
    if not a.image.exists():
        p.error(f"image not found: {a.image}")

    r = Predictor(a.model).predict(a.image)
    print(f"\nImage      : {a.image}")
    print(f"Model      : {r['model']}")
    print(f"Prediction : {r['pred_class']} - {r['pred_full_name']}")
    print(f"Category   : {r['binary_label'].upper()}  (malignant probability {r['malignant_probability']:.1%})")
    print(f"Confidence : {r['confidence']:.2%}\n")
    print("Class confidences:")
    for c, prob in sorted(r["probabilities"].items(), key=lambda kv: -kv[1]):
        print(f"  {c:6s} {config.BINARY_MAP[c]:9s} {prob:7.2%}  {'#' * int(prob * 40)}")

    if a.pdf or a.csv:
        from report import append_csv_row, generate_pdf_report
        out_dir = config.RESULTS_DIR / "reports"
        out_dir.mkdir(parents=True, exist_ok=True)
        if a.pdf:
            pdf = generate_pdf_report(r, out_dir / f"report_{a.image.stem}.pdf")
            print(f"\nPDF report: {pdf}")
        if a.csv:
            print(f"CSV row appended: {append_csv_row(r, out_dir / 'predictions.csv')}")
    print("\nDISCLAIMER: research/educational tool only - not a medical diagnosis.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/report.py
"""PDF and CSV report export for a single prediction.

Usage:
    python src/report.py --image sample_images/<file>.jpg      # writes results/reports/report_<file>.pdf
"""
import argparse
import csv
import io
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
from PIL import Image
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import Image as RLImage
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402

DISCLAIMER = ("DISCLAIMER: This report is produced by an experimental deep-learning system for research and "
              "educational purposes only. It is NOT a medical diagnosis and must not replace examination by a "
              "qualified dermatologist. Any suspicious skin lesion should be assessed by a medical professional.")

CSV_FIELDS = ["timestamp", "filename", "model", "predicted_class", "predicted_name", "category",
              "confidence", "malignant_probability"] + [f"p_{c}" for c in config.CLASS_NAMES]


def _rl_image(arr: np.ndarray, width_cm: float) -> RLImage:
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="PNG")
    buf.seek(0)
    h, w = arr.shape[:2]
    return RLImage(buf, width=width_cm * cm, height=width_cm * cm * h / w)


def result_to_row(result: dict, timestamp: str = None) -> dict:
    row = {
        "timestamp": timestamp or datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "filename": result.get("filename", "uploaded_image"),
        "model": result.get("model", ""),
        "predicted_class": result["pred_class"],
        "predicted_name": result["pred_full_name"],
        "category": result["binary_label"],
        "confidence": round(result["confidence"], 4),
        "malignant_probability": round(result["malignant_probability"], 4),
    }
    row.update({f"p_{c}": round(p, 4) for c, p in result["probabilities"].items()})
    return row


def csv_bytes(rows) -> bytes:
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=CSV_FIELDS)
    w.writeheader()
    w.writerows(rows)
    return buf.getvalue().encode("utf-8")


def append_csv_row(result: dict, path: Path) -> Path:
    path = Path(path)
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if new:
            w.writeheader()
        w.writerow(result_to_row(result))
    return path


def generate_pdf_report(result: dict, out_path=None) -> object:
    """Build the PDF. Writes to out_path if given and returns the path, else returns the PDF bytes."""
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=A4, leftMargin=1.8 * cm, rightMargin=1.8 * cm,
                            topMargin=1.5 * cm, bottomMargin=1.5 * cm,
                            title="Skin Lesion Analysis Report")
    ss = getSampleStyleSheet()
    small = ParagraphStyle("small", parent=ss["Normal"], fontSize=8, leading=10, textColor=colors.HexColor("#555555"))
    is_mal = result["binary_label"] == "Malignant"
    badge_color = colors.HexColor("#c0392b") if is_mal else colors.HexColor("#1e8449")
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    story = [
        Paragraph("Skin Lesion Analysis Report", ss["Title"]),
        Paragraph("Deep Learning-Based Skin Cancer Detection and Classification "
                  "Using Dermoscopic Image Analysis", ss["Italic"]),
        Spacer(1, 0.3 * cm),
    ]
    meta = Table([
        ["Generated", ts], ["Image", result.get("filename", "uploaded_image")],
        ["Model", result.get("model", "")],
    ], colWidths=[3.5 * cm, 13.5 * cm])
    meta.setStyle(TableStyle([("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"), ("FONTSIZE", (0, 0), (-1, -1), 9)]))
    story += [meta, Spacer(1, 0.3 * cm)]

    summary = Table([
        ["Predicted class", f"{result['pred_class']}  -  {result['pred_full_name']}"],
        ["Category", result["binary_label"].upper()],
        ["Confidence", f"{result['confidence']:.2%}"],
        ["Malignant probability", f"{result['malignant_probability']:.2%}"],
    ], colWidths=[4.5 * cm, 12.5 * cm])
    summary.setStyle(TableStyle([
        ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"),
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#f4f6f7")),
        ("TEXTCOLOR", (1, 1), (1, 1), badge_color), ("FONTNAME", (1, 1), (1, 1), "Helvetica-Bold"),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.grey), ("INNERGRID", (0, 0), (-1, -1), 0.3, colors.lightgrey),
        ("FONTSIZE", (0, 0), (-1, -1), 10), ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))
    story += [summary, Spacer(1, 0.4 * cm)]

    w = 5.4
    imgs = Table([[_rl_image(result["original"], w), _rl_image(result["preprocessed"], w),
                   _rl_image(result["overlay"], w)],
                  ["Original", "Preprocessed", "Grad-CAM heatmap"]], colWidths=[5.7 * cm] * 3)
    imgs.setStyle(TableStyle([("ALIGN", (0, 0), (-1, -1), "CENTER"), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                              ("FONTSIZE", (0, 1), (-1, 1), 9)]))
    story += [imgs, Spacer(1, 0.2 * cm),
              Paragraph("Grad-CAM highlights (red/yellow) the image regions that most influenced the "
                        "predicted class.", small), Spacer(1, 0.4 * cm)]

    rows = [["Class", "Description", "Type", "Confidence"]]
    for c, p in sorted(result["probabilities"].items(), key=lambda kv: -kv[1]):
        rows.append([c, config.CLASS_FULL_NAMES[c], config.BINARY_MAP[c], f"{p:.2%}"])
    conf = Table(rows, colWidths=[2 * cm, 9 * cm, 3 * cm, 3 * cm])
    conf.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2c3e50")), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"), ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f2f3f4")]),
        ("BACKGROUND", (0, 1), (-1, 1), colors.HexColor("#fdebd0")),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.grey), ("ALIGN", (3, 0), (3, -1), "RIGHT"),
    ]))
    story += [Paragraph("Confidence scores (all classes)", ss["Heading3"]), conf, Spacer(1, 0.6 * cm),
              Paragraph(DISCLAIMER, ParagraphStyle("disc", parent=small, textColor=colors.HexColor("#922b21"),
                                                   borderColor=colors.HexColor("#922b21"), borderWidth=0.6,
                                                   borderPadding=6, fontSize=8.5, leading=11))]
    doc.build(story)
    pdf = buf.getvalue()
    if out_path is None:
        return pdf
    Path(out_path).write_bytes(pdf)
    return Path(out_path)


if __name__ == "__main__":
    from predict import Predictor

    p = argparse.ArgumentParser(description="Generate a PDF + CSV report for one image")
    p.add_argument("--image", type=Path, required=True)
    p.add_argument("--model", default=config.DEFAULT_MODEL, choices=config.AVAILABLE_MODELS)
    a = p.parse_args()
    out_dir = config.RESULTS_DIR / "reports"
    out_dir.mkdir(parents=True, exist_ok=True)
    res = Predictor(a.model).predict(a.image)
    print("PDF:", generate_pdf_report(res, out_dir / f"report_{a.image.stem}.pdf"))
    print("CSV:", append_csv_row(res, out_dir / "predictions.csv"))

In [ ]:
%%writefile src/compare_models.py
"""Train and evaluate EfficientNet-B0, ResNet50 and DenseNet121, then compare them.

Outputs: results/model_comparison.csv and results/model_comparison.png

Usage:
    python src/compare_models.py --epochs 10                 # train + evaluate all three
    python src/compare_models.py --reuse                     # reuse existing checkpoints where present
    python src/compare_models.py --quick                     # small subset, 2 epochs (pipeline check)
"""
import argparse
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from evaluate import evaluate  # noqa: E402
from models import DISPLAY_NAMES, build_model, count_parameters  # noqa: E402
from train import train  # noqa: E402

METRIC_COLS = [("accuracy", "Accuracy"), ("balanced_accuracy", "Balanced acc."), ("macro_f1", "Macro F1"),
               ("macro_roc_auc", "Macro ROC-AUC"), ("bin_sensitivity", "Malignant sensitivity"),
               ("bin_specificity", "Malignant specificity")]


def plot_comparison(df: pd.DataFrame, out: Path, title: str):
    x = np.arange(len(METRIC_COLS))
    width = 0.8 / len(df)
    palette = ["#2a78b8", "#e08a1e", "#3a9d5d"]
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, (_, row) in enumerate(df.iterrows()):
        vals = [row[k] for k, _ in METRIC_COLS]
        bars = ax.bar(x + (i - (len(df) - 1) / 2) * width, vals, width, label=row["model"],
                      color=palette[i % len(palette)])
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x, [label for _, label in METRIC_COLS])
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score (test set)")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


def main():
    p = argparse.ArgumentParser(description="Compare the three transfer-learning backbones")
    p.add_argument("--models", nargs="+", default=config.AVAILABLE_MODELS, choices=config.AVAILABLE_MODELS)
    p.add_argument("--epochs", type=int, default=config.EPOCHS)
    p.add_argument("--batch-size", type=int, default=config.BATCH_SIZE)
    p.add_argument("--quick", action="store_true")
    p.add_argument("--reuse", action="store_true", help="skip training when a checkpoint already exists")
    a = p.parse_args()

    suffix = "_quick" if a.quick else ""
    rows = []
    for name in a.models:
        ckpt = config.model_path(name, a.quick)
        if not (a.reuse and ckpt.exists()):
            train(name, a.epochs, a.batch_size, quick=a.quick)
        else:
            print(f"Reusing existing checkpoint {ckpt.name}")
        m = evaluate(name, quick=a.quick, batch_size=a.batch_size)
        summ_path = config.RESULTS_DIR / f"{name}{suffix}" / "training_summary.json"
        summ = json.loads(summ_path.read_text()) if summ_path.exists() else {}
        rows.append({
            "model": DISPLAY_NAMES[name],
            "params_millions": round(count_parameters(build_model(name, pretrained=False)) / 1e6, 2),
            **{k: round(v, 4) for k, v in m["overall"].items() if k != "n_test"},
            **{f"bin_{k}": round(v, 4) for k, v in m["binary"].items() if k != "confusion_matrix"},
            "epochs_run": summ.get("epochs_run"),
            "best_epoch": m.get("best_epoch"),
            "train_minutes": round(summ.get("train_minutes", float("nan")), 1),
        })

    df = pd.DataFrame(rows)
    out_csv = config.RESULTS_DIR / f"model_comparison{suffix}.csv"
    df.to_csv(out_csv, index=False)
    plot_comparison(df, config.RESULTS_DIR / f"model_comparison{suffix}.png",
                    "Model comparison on HAM10000 test set" + (" (quick mode)" if a.quick else ""))
    print(df.to_string(index=False))
    print(f"Saved {out_csv}")


if __name__ == "__main__":
    main()

## 2. Kaggle credentials and download
Create a token at <https://www.kaggle.com/settings> → *API* → *Generate New Token*.
Either paste the **new-format token** when prompted, or upload a legacy **kaggle.json**.

In [ ]:
import os, getpass, pathlib
kdir = pathlib.Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
if not ((kdir / "access_token").exists() or (kdir / "kaggle.json").exists() or os.environ.get("KAGGLE_API_TOKEN")):
    choice = input("Type 't' to paste an API token, or 'j' to upload kaggle.json: ").strip().lower()
    if choice == "j":
        from google.colab import files
        up = files.upload()
        (kdir / "kaggle.json").write_bytes(up["kaggle.json"]); os.chmod(kdir / "kaggle.json", 0o600)
    else:
        (kdir / "access_token").write_text(getpass.getpass("Kaggle API token: ").strip())
        os.chmod(kdir / "access_token", 0o600)

In [ ]:
!python scripts/download_data.py

## 3. Exploratory data analysis

In [ ]:
import sys; sys.path.insert(0, "."); sys.path.insert(0, "src")
import matplotlib.pyplot as plt, seaborn as sns, pandas as pd
import config
from dataset import load_metadata
df = load_metadata()
print(df.shape, "| unique lesions:", df.lesion_id.nunique())
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
counts = df.dx.value_counts().reindex(config.CLASS_NAMES)
colors = ["#c0392b" if config.BINARY_MAP[c] == "Malignant" else "#1e8449" for c in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=colors); axes[0].bar_label(bars)
axes[0].set_title("Class distribution (red = malignant, green = benign)"); axes[0].set_ylabel("images")
df.binary.value_counts().plot.pie(ax=axes[1], autopct="%1.1f%%", colors=["#1e8449", "#c0392b"])
axes[1].set_title("Benign vs malignant"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.histplot(df, x="age", hue="binary", bins=20, multiple="stack", ax=axes[0]); axes[0].set_title("Age")
df.sex.value_counts().plot.bar(ax=axes[1], color="#5d6d7e"); axes[1].set_title("Sex")
df.localization.value_counts().plot.barh(ax=axes[2], color="#5d6d7e"); axes[2].set_title("Localization")
plt.tight_layout(); plt.show()

In [ ]:
from preprocess import load_image_rgb, remove_hair, apply_clahe, preprocess_image
fig, axes = plt.subplots(len(config.CLASS_NAMES), 4, figsize=(12, 3 * len(config.CLASS_NAMES)))
for r, c in enumerate(config.CLASS_NAMES):
    img = load_image_rgb(df[df.dx == c].iloc[0].path)
    no_hair = remove_hair(img)
    for ax, im, t in zip(axes[r], [img, no_hair, apply_clahe(no_hair), preprocess_image(img)],
                         ["original", "hair removed", "CLAHE", "final 224x224"]):
        ax.imshow(im); ax.set_title(f"{c}: {t}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Leakage-free split (by `lesion_id`) and preprocessing cache

In [ ]:
!python src/dataset.py
!python src/preprocess.py

## 5. Training
First a `--quick` run to verify the pipeline, then the full EfficientNet-B0 training (15 epochs, early stopping).

In [ ]:
!python src/train.py --quick && python src/evaluate.py --quick

In [ ]:
!python src/train.py --model efficientnet_b0 --epochs 15

## 6. Evaluation on the held-out test set

In [ ]:
!python src/evaluate.py --model efficientnet_b0
from IPython.display import Image, display
for f in ["training_curves.png", "confusion_matrix.png", "confusion_matrix_binary.png", "roc_curves.png"]:
    display(Image(f"results/efficientnet_b0/{f}"))

## 7. Grad-CAM explanations

In [ ]:
!python scripts/prepare_samples.py
!python src/gradcam.py --all-samples
import glob
for f in sorted(glob.glob("results/gradcam/*.png")):
    display(Image(f))

In [ ]:
!python src/predict.py --image $(ls sample_images/*.jpg | head -1) --pdf --csv

## 8. Model comparison (EfficientNet-B0 vs ResNet50 vs DenseNet121)

In [ ]:
!python src/compare_models.py --epochs 15 --reuse
display(pd.read_csv("results/model_comparison.csv"))
display(Image("results/model_comparison.png"))

## 9. Download trained models and results

In [ ]:
!zip -qr trained_outputs.zip models results sample_images
try:
    from google.colab import files; files.download("trained_outputs.zip")
except ImportError:
    print("Saved trained_outputs.zip")